# APD Vision — fine-tuning gloves dan shoes sekitar dua jam

Notebook ini melakukan **satu pilot terkontrol** dari model aktif `best.pt`, bukan dari pretrained generik. Tujuannya menaikkan `gloves` dan `shoes` sambil memasang lantai untuk `helmet` dan `vest`.

Strategi data default: minimal 65% gambar train full-PPE sebagai jangkar dan maksimal 35% specialist gloves/shoes. Validation hanya berasal dari full-PPE. Gold reviewed 340 gambar tidak pernah menjadi train.

Output selalu bernama `candidate`; notebook tidak pernah mengganti model aktif. Aktifkan **Runtime > Change runtime type > T4 GPU**, lalu jalankan sel secara berurutan.

## 1. Periksa GPU dan pasang dependensi

In [ ]:
import subprocess, sys
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3,<9', 'roboflow>=1.1', 'pyyaml>=6.0', 'pillow>=10'], check=True)

## 2. Siapkan tool repository

Unggah `ppe-colab-tools.zip` dari root repository ke `/content` melalui panel Files. ZIP hanya memuat tool training yang diperlukan; token GitHub dan clone repository tidak dibutuhkan.

In [ ]:
import os, shutil, stat, sys, zipfile
from pathlib import Path

TOOLS_ZIP = Path('/content/ppe-colab-tools.zip').resolve()
EXTRACT_ROOT = Path('/content/colab_tools').resolve()
if not TOOLS_ZIP.is_file():
    raise RuntimeError('Unggah ppe-colab-tools.zip ke /content terlebih dahulu.')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(TOOLS_ZIP) as archive:
    for member in archive.infolist():
        destination = (EXTRACT_ROOT / member.filename).resolve()
        if destination != EXTRACT_ROOT and EXTRACT_ROOT not in destination.parents:
            raise RuntimeError(f'Path ZIP tidak aman: {member.filename}')
        if stat.S_ISLNK(member.external_attr >> 16):
            raise RuntimeError(f'Symlink dalam ZIP tidak diizinkan: {member.filename}')
    archive.extractall(EXTRACT_ROOT)
project_candidates = sorted({p.parent.parent.resolve() for p in EXTRACT_ROOT.rglob('training/merge_datasets.py') if (p.parent/'fetch_datasets.py').is_file() and (p.parent/'evaluate_model.py').is_file()})
if len(project_candidates) != 1:
    raise RuntimeError(f'Isi ZIP tool tidak lengkap atau ambigu: {project_candidates}')
PROJECT_DIR = project_candidates[0]
TRAINING_DIR = PROJECT_DIR/'training'
for name in ('fetch_datasets.py', 'merge_datasets.py', 'evaluate_model.py'):
    compile((TRAINING_DIR/name).read_text(), name, 'exec')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('Tool siap:', TRAINING_DIR)

## 3. Secret, Drive, dan konfigurasi kandidat

Simpan API key sebagai Colab Secret `ROBOFLOW_API_KEY`. Periksa sendiri lisensi seluruh sumber aktif pada tautan Universe, lalu ubah `LICENSES_CONFIRMED` menjadi `True`. Jangan menyalin API key ke notebook.

Daftar alternatif lengkap dan alasan pemilihannya ada di `training/DATASET_CANDIDATES_GLOVES_SHOES.md`.

In [ ]:
from datetime import datetime, timezone
from google.colab import drive, userdata

TARGET = ['helmet', 'vest', 'gloves', 'shoes']
EXPECTED_ACTIVE_SHA256 = '8d16dff9095a104d7e11e78bcd21f1e9d4517f536e5e3b7648650bf0b69c3f5e'
SOURCE_CONFIG = [
    # Jangkar empat kelas; wajib anotasi lengkap pada objek yang terlihat.
    {'role': 'full', 'spec': 'akfa-beqxl/safety-rd-v1', 'enabled': True},
    # Alternatif full-PPE: aktifkan hanya setelah audit visual dan kelas.
    {'role': 'full', 'spec': 'skcet-g4h72/construction-ppe-rdhzo', 'enabled': False},
    # Specialist nonaktif setelah audit preview. Aktifkan hanya jika anotasi/domain sudah diperbaiki.
    {'role': 'gloves', 'spec': 'construction-images/gloves-qx5sc/2', 'enabled': False},
    # Nama proyek menyesatkan: kelas aktual hanya `Boots`; perannya shoes.
    {'role': 'shoes', 'spec': 'test-pkubl/gloves-xjuja/2', 'enabled': False},
    {'role': 'shoes', 'spec': 'bns218/safety-shoes-yawyh', 'enabled': False},
    {'role': 'shoes', 'spec': 'nedrick-chandra-gpg1l/safety-shoes-detection-5qgkg', 'enabled': False},
]
LICENSES_CONFIRMED = False  # Ubah hanya setelah lisensi semua sumber aktif diperiksa.
MAX_FULL_TRAIN_IMAGES = 5000
SPECIALIST_TOTAL_FRACTION = 0.35
SEED = 42
EPOCHS = 25
IMGSZ = 960
TRAIN_TIME_HOURS = 1.35

api_key = userdata.get('ROBOFLOW_API_KEY')
if not api_key:
    raise RuntimeError('Secret ROBOFLOW_API_KEY belum tersedia atau aksesnya belum diaktifkan.')
os.environ['ROBOFLOW_API_KEY'] = api_key
drive.mount('/content/drive')
RUN_ID = datetime.now(timezone.utc).strftime('gloves_shoes_%Y%m%dT%H%M%SZ')
RUN_DIR = Path('/content/drive/MyDrive/APD_Vision/runs')/RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
active_sources = [row for row in SOURCE_CONFIG if row['enabled']]
if not any(row['role'] == 'full' for row in active_sources):
    raise RuntimeError('Minimal satu jangkar full-PPE harus aktif.')
if any('sdp-lfigk/ppe-detection-ozhfb' in row['spec'] for row in active_sources):
    raise RuntimeError('Hold-out lama dilarang menjadi train.')
print('Run:', RUN_ID)
print('Sumber aktif:')
for row in active_sources:
    print(f"- {row['role']}: {row['spec']}")

## 4. Unduh sumber dan audit label

Sel ini berhenti sampai konfirmasi lisensi diberikan. Setelah download, audit memeriksa pemetaan kelas, jumlah gambar/instance, label hilang/rusak, dan exact duplicate. Sumber full-PPE harus benar-benar memiliki instance keempat kelas.

In [ ]:
if not LICENSES_CONFIRMED:
    raise RuntimeError('Periksa lisensi semua tautan aktif, lalu set LICENSES_CONFIRMED = True.')

import hashlib, json, random
from collections import Counter, defaultdict
import yaml
from roboflow import Roboflow
from training.fetch_datasets import fetch
from training.merge_datasets import CLASS_MAP, IGNORE, names_of, split_dirs

RAW_ROOT = Path('/content/datasets/raw').resolve()
if RAW_ROOT.exists():
    shutil.rmtree(RAW_ROOT)
RAW_ROOT.mkdir(parents=True)
rf = Roboflow(api_key=api_key)
RAW_SOURCES = []
role_index = Counter()
for source in active_sources:
    role = source['role']
    destination = RAW_ROOT/f'{role}_{role_index[role]}'
    role_index[role] += 1
    location = Path(fetch(source['spec'], str(destination), rf)).resolve()
    RAW_SOURCES.append({**source, 'path': location})

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
audit = {'run_id': RUN_ID, 'sources': [], 'duplicate_hashes': {}}
hash_locations = defaultdict(list)
for source in RAW_SOURCES:
    root = source['path']
    names = names_of(root)
    mapped = {i: CLASS_MAP[name.lower()] for i, name in enumerate(names) if name.lower() in CLASS_MAP}
    unknown = [name for i, name in enumerate(names) if i not in mapped and name.lower().strip() not in IGNORE]
    images_per_split, instances, malformed, missing_labels = Counter(), Counter(), 0, 0
    for split, image_dir, label_dir in split_dirs(root):
        for image_path in sorted(p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES):
            images_per_split[split] += 1
            digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
            hash_locations[digest].append(f"{source['spec']}:{split}:{image_path.name}")
            label_path = label_dir/f'{image_path.stem}.txt'
            if not label_path.exists():
                missing_labels += 1
                continue
            for line in label_path.read_text().splitlines():
                parts = line.split()
                try:
                    class_id = int(parts[0])
                    coords = [float(value) for value in parts[1:5]]
                    if len(parts) < 5 or class_id < 0 or class_id >= len(names) or any(not 0 <= value <= 1 for value in coords):
                        raise ValueError
                    if class_id in mapped:
                        instances[mapped[class_id]] += 1
                except (ValueError, IndexError):
                    malformed += 1
    metadata = yaml.safe_load((root/'data.yaml').read_text()) or {}
    license_value = metadata.get('license') or (metadata.get('roboflow') or {}).get('license') or 'Tidak tercantum di data.yaml'
    record = {
        'spec': source['spec'], 'role': source['role'], 'path': str(root), 'license_metadata': license_value,
        'raw_names': names, 'mapped_classes': sorted(set(mapped.values())), 'unknown_classes': unknown,
        'images_per_split': dict(images_per_split), 'instances': dict(instances),
        'missing_labels': missing_labels, 'malformed_labels': malformed,
    }
    audit['sources'].append(record)
    required = set(TARGET) if source['role'] == 'full' else {source['role']}
    if not required.issubset({name for name, count in instances.items() if count > 0}):
        raise RuntimeError(f"Cakupan label {source['spec']} tidak memenuhi peran {source['role']}: {dict(instances)}")
    if malformed:
        raise RuntimeError(f"Label rusak ditemukan pada {source['spec']}: {malformed}")
    print(json.dumps(record, indent=2, ensure_ascii=False))
audit['duplicate_hashes'] = {digest: paths for digest, paths in hash_locations.items() if len(paths) > 1}
AUDIT_PATH = RUN_DIR/'dataset_audit.json'
AUDIT_PATH.write_text(json.dumps(audit, indent=2, ensure_ascii=False))
print('Exact duplicate lintas lokasi:', len(audit['duplicate_hashes']))
print('Audit tersimpan:', AUDIT_PATH)

## 5. Preview anotasi dan persetujuan visual

Tinjau box dari sampel setiap sumber. Pastikan full-PPE menampilkan pekerja utuh/CCTV dan semua APD yang terlihat berlabel. Nonaktifkan specialist yang dominan close-up produk atau sering menampilkan APD lain tanpa label. Setelah mengubah sumber, ulangi dari sel konfigurasi.

In [ ]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

random.seed(SEED)
preview_rows = []
for source in RAW_SOURCES:
    candidates = []
    for split, image_dir, label_dir in split_dirs(source['path']):
        if split == 'train':
            candidates.extend((path, label_dir/path.with_suffix('.txt').name) for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
    for image_path, label_path in random.sample(candidates, min(3, len(candidates))):
        image = Image.open(image_path).convert('RGB')
        draw = ImageDraw.Draw(image)
        names = names_of(source['path'])
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                class_id = int(parts[0]); x, y, w, h = map(float, parts[1:5]); width, height = image.size
                box = ((x-w/2)*width, (y-h/2)*height, (x+w/2)*width, (y+h/2)*height)
                draw.rectangle(box, outline='red', width=max(2, width//400))
                draw.text((box[0], max(0, box[1]-14)), names[class_id], fill='red')
        preview_rows.append((source['spec'], image))
fig, axes = plt.subplots(len(RAW_SOURCES), 3, figsize=(16, 5*len(RAW_SOURCES)))
axes = axes.reshape(len(RAW_SOURCES), 3)
for row_index, source in enumerate(RAW_SOURCES):
    items = [(name, image) for name, image in preview_rows if name == source['spec']]
    for column in range(3):
        axes[row_index, column].axis('off')
        if column < len(items):
            axes[row_index, column].imshow(items[column][1])
            axes[row_index, column].set_title(source['spec'], fontsize=9)
plt.tight_layout()
PREVIEW_PATH = RUN_DIR/'dataset_preview.jpg'
fig.savefig(PREVIEW_PATH, dpi=130)
plt.show()
print('Preview tersimpan:', PREVIEW_PATH)

## 6. Normalisasi dan bentuk campuran dengan specialist maksimal 35%

Ubah `VISUAL_AUDIT_CONFIRMED` setelah preview benar-benar diperiksa. Default konservatif hanya memakai full-PPE; specialist harus tetap nonaktif bila APD lain yang terlihat tidak dianotasi atau domainnya tidak sesuai. Semua sumber aktif dinormalisasi ke urutan `helmet`, `vest`, `gloves`, `shoes`. Validation hanya dari full-PPE. Exact duplicate dihapus saat komposisi dibuat.

In [ ]:
VISUAL_AUDIT_CONFIRMED = False  # Ubah menjadi True setelah preview diperiksa.
if not VISUAL_AUDIT_CONFIRMED:
    raise RuntimeError('Periksa preview dan kelengkapan anotasi, lalu set VISUAL_AUDIT_CONFIRMED = True.')

MERGED_ROOT = Path('/content/datasets/merged').resolve()
BALANCED_ROOT = Path('/content/datasets/ppe4_balanced').resolve()
for path in (MERGED_ROOT, BALANCED_ROOT):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True)
merged_by_role = {}
for role in ('full', 'gloves', 'shoes'):
    paths = [str(source['path']) for source in RAW_SOURCES if source['role'] == role]
    if not paths:
        continue
    output = MERGED_ROOT/role
    subprocess.run([sys.executable, str(TRAINING_DIR/'merge_datasets.py'), '--src', *paths, '--out', str(output)], check=True)
    merged_by_role[role] = output

for split in ('train', 'val'):
    (BALANCED_ROOT/'images'/split).mkdir(parents=True)
    (BALANCED_ROOT/'labels'/split).mkdir(parents=True)

def image_pairs(root, split):
    image_dir, label_dir = root/'images'/split, root/'labels'/split
    if not image_dir.exists():
        return []
    return [(path, label_dir/f'{path.stem}.txt') for path in sorted(image_dir.iterdir()) if path.suffix.lower() in IMAGE_SUFFIXES]

def copy_selected(pairs, split, prefix, limit, used_hashes):
    candidates = list(pairs)
    random.Random(SEED + sum(map(ord, prefix))).shuffle(candidates)
    copied = []
    for image_path, label_path in candidates:
        if len(copied) >= limit:
            break
        digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
        if digest in used_hashes:
            continue
        used_hashes.add(digest)
        stem = f'{prefix}__{image_path.stem}'
        shutil.copy2(image_path, BALANCED_ROOT/'images'/split/f'{stem}{image_path.suffix.lower()}')
        (BALANCED_ROOT/'labels'/split/f'{stem}.txt').write_text(label_path.read_text() if label_path.exists() else '')
        copied.append(digest)
    return copied

used_hashes = set()
full_val = image_pairs(merged_by_role['full'], 'val')
if not full_val:
    raise RuntimeError('Jangkar full-PPE tidak memiliki validation split.')
validation_hashes = copy_selected(full_val, 'val', 'full', min(1500, len(full_val)), used_hashes)
full_train = image_pairs(merged_by_role['full'], 'train')
selected_full = copy_selected(full_train, 'train', 'full', min(MAX_FULL_TRAIN_IMAGES, len(full_train)), used_hashes)
if not selected_full:
    raise RuntimeError('Train full-PPE kosong setelah deduplikasi.')
per_specialist_cap = int(len(selected_full) * (SPECIALIST_TOTAL_FRACTION/2) / (1-SPECIALIST_TOTAL_FRACTION))
selected_by_role = {'full': selected_full}
for role in ('gloves', 'shoes'):
    pairs = image_pairs(merged_by_role[role], 'train') if role in merged_by_role else []
    selected_by_role[role] = copy_selected(pairs, 'train', role, min(per_specialist_cap, len(pairs)), used_hashes)

(BALANCED_ROOT/'data.yaml').write_text(yaml.safe_dump({
    'path': str(BALANCED_ROOT), 'train': 'images/train', 'val': 'images/val', 'names': dict(enumerate(TARGET)),
}, sort_keys=False))

def count_instances(split):
    counts = Counter()
    for label_path in (BALANCED_ROOT/'labels'/split).glob('*.txt'):
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if parts:
                counts[TARGET[int(parts[0])]] += 1
    return dict(counts)
composition = {
    'train_images': {role: len(items) for role, items in selected_by_role.items()},
    'validation_images': len(validation_hashes),
    'train_instances': count_instances('train'), 'validation_instances': count_instances('val'),
}
train_total = sum(composition['train_images'].values())
composition['train_fraction'] = {role: round(count/train_total, 4) for role, count in composition['train_images'].items()}
if any(composition['validation_instances'].get(name, 0) == 0 for name in TARGET):
    raise RuntimeError(f"Validation full-PPE tidak mencakup semua kelas: {composition['validation_instances']}")
COMPOSITION_PATH = RUN_DIR/'dataset_composition.json'
COMPOSITION_PATH.write_text(json.dumps(composition, indent=2))
SELECTED_DATA_HASHES = used_hashes.copy()
print(json.dumps(composition, indent=2))
print('Dataset siap:', BALANCED_ROOT/'data.yaml')

## 7. Unggah model aktif sebagai initial weights

Unggah salinan `backend/weights/best.pt` melalui panel Files dengan nama `/content/active_best.pt`. Hash dan urutan empat kelas dicatat. Jangan gunakan candidate eksperimen lama sebagai initial weights.

In [ ]:
from ultralytics import YOLO

ACTIVE_WEIGHTS = Path('/content/active_best.pt').resolve()
if not ACTIVE_WEIGHTS.is_file():
    raise RuntimeError('Unggah backend/weights/best.pt sebagai /content/active_best.pt.')
active_sha256 = hashlib.sha256(ACTIVE_WEIGHTS.read_bytes()).hexdigest()
if active_sha256 != EXPECTED_ACTIVE_SHA256:
    raise RuntimeError(f'Hash bobot tidak cocok dengan model aktif yang diaudit: {active_sha256}')
baseline_model = YOLO(str(ACTIVE_WEIGHTS))
baseline_names = list(baseline_model.names.values())
if baseline_names != TARGET:
    raise RuntimeError(f'Urutan kelas model aktif salah: {baseline_names}; diharapkan {TARGET}.')
weights_record = {'path': str(ACTIVE_WEIGHTS), 'sha256': active_sha256, 'names': baseline_names}
(RUN_DIR/'initial_weights.json').write_text(json.dumps(weights_record, indent=2))
shutil.copy2(ACTIVE_WEIGHTS, RUN_DIR/'baseline_active_uploaded.pt')
del baseline_model
print(json.dumps(weights_record, indent=2))

## 8. Fine-tune maksimal 25 epoch dengan batas waktu

Konfigurasi: `imgsz=960`, `batch=-1`, `patience=7`, `freeze=10`, `lr0=0.001`, `close_mosaic=5`. Callback menghentikan training antar-batch setelah 1,35 jam agar sisa jatah dua jam tersedia untuk evaluasi baseline dan candidate. Parameter Ultralytics `time` sengaja tidak dipakai karena parameter itu menghitung ulang jumlah epoch. Checkpoint berada di Google Drive.

In [ ]:
import time

model = YOLO(str(ACTIVE_WEIGHTS))
training_started = time.monotonic()
training_deadline = training_started + TRAIN_TIME_HOURS*3600
def stop_training_at_deadline(trainer):
    if time.monotonic() >= training_deadline:
        trainer.stop = True
model.add_callback('on_train_batch_end', stop_training_at_deadline)
model.train(
    data=str(BALANCED_ROOT/'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ,
    batch=-1, patience=7, freeze=10, lr0=0.001, close_mosaic=5, cos_lr=True,
    device=0, seed=SEED, deterministic=True, project=str(RUN_DIR), name='train',
    exist_ok=True, save=True, save_period=5, plots=True, cache=False, workers=4,
)
CANDIDATE = RUN_DIR/'train'/'weights'/'best.pt'
if not CANDIDATE.is_file():
    raise RuntimeError(f'Candidate tidak ditemukan: {CANDIDATE}')
candidate_names = list(YOLO(str(CANDIDATE)).names.values())
if candidate_names != TARGET:
    raise RuntimeError(f'Urutan kelas candidate berubah: {candidate_names}')
CANDIDATE_OUTPUT = RUN_DIR/f'candidate_{RUN_ID}.pt'
shutil.copy2(CANDIDATE, CANDIDATE_OUTPUT)
print('Candidate tersimpan:', CANDIDATE_OUTPUT)

## 9. Evaluasi internal

Metrik validation internal hanya untuk diagnosis. Ia bukan gate karena domain dan sumbernya masih berkaitan dengan train.

In [ ]:
INTERNAL_METRICS = RUN_DIR/'metrics_internal_val.json'
subprocess.run([
    sys.executable, str(TRAINING_DIR/'evaluate_model.py'), '--weights', str(CANDIDATE),
    '--data', str(BALANCED_ROOT/'data.yaml'), '--split', 'val', '--imgsz', str(IMGSZ),
    '--device', '0', '--json-out', str(INTERNAL_METRICS),
], check=True)

## 10. Evaluasi baseline dan candidate pada gold reviewed 340 gambar

Buat ZIP dari folder `datasets/gold/` lokal sehingga ZIP memuat `manifest.jsonl`, `data.yaml`, `images/test`, dan `labels/test`. Unggah sebagai `/content/gold_reviewed_340.zip`. Sel memeriksa manifest, jumlah 340 gambar, path traversal, symlink, dan overlap hash dengan data train.

Gate pilot: `helmet >= 0,55`, `vest >= 0,38`, `gloves >= 0,15`, `shoes >= 0,15`. Gate ini hanya memilih candidate untuk review lanjutan dan **bukan** gate kelayakan pelanggaran otomatis.

In [ ]:
GOLD_ZIP = Path('/content/gold_reviewed_340.zip').resolve()
if not GOLD_ZIP.is_file():
    raise RuntimeError('Unggah ZIP datasets/gold sebagai /content/gold_reviewed_340.zip.')
GOLD_ROOT = Path('/content/gold_reviewed_340').resolve()
if GOLD_ROOT.exists():
    shutil.rmtree(GOLD_ROOT)
GOLD_ROOT.mkdir()
with zipfile.ZipFile(GOLD_ZIP) as archive:
    for member in archive.infolist():
        destination = (GOLD_ROOT/member.filename).resolve()
        if destination != GOLD_ROOT and GOLD_ROOT not in destination.parents:
            raise RuntimeError(f'Path ZIP gold tidak aman: {member.filename}')
        if stat.S_ISLNK(member.external_attr >> 16):
            raise RuntimeError(f'Symlink dalam ZIP gold tidak diizinkan: {member.filename}')
    archive.extractall(GOLD_ROOT)
manifests = list(GOLD_ROOT.rglob('manifest.jsonl'))
if len(manifests) != 1:
    raise RuntimeError(f'ZIP harus memuat tepat satu manifest.jsonl; ditemukan {len(manifests)}.')
GOLD_DIR = manifests[0].parent.resolve()
records = [json.loads(line) for line in manifests[0].read_text().splitlines() if line.strip()]
expected_hashes = {row['sha256'] for row in records}
gold_images = [path for path in GOLD_DIR.rglob('*') if path.suffix.lower() in IMAGE_SUFFIXES]
actual_hashes = {hashlib.sha256(path.read_bytes()).hexdigest() for path in gold_images}
if len(records) != 340 or len(gold_images) != 340 or len(actual_hashes) != 340 or actual_hashes != expected_hashes:
    raise RuntimeError('Gold bukan set reviewed aktif 340 gambar atau integritas manifest gagal.')
leaks = actual_hashes & SELECTED_DATA_HASHES
if leaks:
    raise RuntimeError(f'DATA LEAKAGE: {len(leaks)} gambar gold terdapat pada train/validation terpilih.')
GOLD_YAML = GOLD_DIR/'data.yaml'
gold_config = yaml.safe_load(GOLD_YAML.read_text())
gold_config['path'] = str(GOLD_DIR)
GOLD_YAML.write_text(yaml.safe_dump(gold_config, sort_keys=False))

BASELINE_GOLD = RUN_DIR/'metrics_gold_baseline.json'
CANDIDATE_GOLD = RUN_DIR/'metrics_gold_candidate.json'
for weights, output in ((ACTIVE_WEIGHTS, BASELINE_GOLD), (CANDIDATE, CANDIDATE_GOLD)):
    subprocess.run([
        sys.executable, str(TRAINING_DIR/'evaluate_model.py'), '--weights', str(weights),
        '--data', str(GOLD_YAML), '--split', 'test', '--imgsz', str(IMGSZ),
        '--device', '0', '--json-out', str(output),
    ], check=True)
baseline = json.loads(BASELINE_GOLD.read_text())
candidate = json.loads(CANDIDATE_GOLD.read_text())
thresholds = {'helmet': 0.55, 'vest': 0.38, 'gloves': 0.15, 'shoes': 0.15}
problems = [f"{name} {candidate['per_class'].get(name)} < {minimum}" for name, minimum in thresholds.items() if candidate['per_class'].get(name, -1) < minimum]
comparison = {
    'status': 'LOLOS GATE PILOT' if not problems else 'GAGAL GATE PILOT',
    'gate_scope': 'Eksperimen; bukan gate produksi atau aktivasi pelanggaran otomatis.',
    'thresholds': thresholds, 'problems': problems,
    'baseline_historical': {'helmet': 0.604, 'vest': 0.432, 'gloves': 0.045, 'shoes': 0.026},
    'baseline_reproduced': baseline, 'candidate': candidate,
    'delta_per_class': {name: round(candidate['per_class'][name]-baseline['per_class'][name], 4) for name in TARGET},
}
COMPARISON_PATH = RUN_DIR/'gold_comparison.json'
COMPARISON_PATH.write_text(json.dumps(comparison, indent=2, ensure_ascii=False))
print(json.dumps(comparison, indent=2, ensure_ascii=False))
print('Model aktif tidak diubah. Candidate tetap harus direview manual.')

## 11. Unduh artefak

Semua artefak tetap tersimpan di `MyDrive/APD_Vision/runs/<RUN_ID>`. Unduh candidate dan bukti evaluasi untuk dipindahkan ke mesin lokal. Jangan menamai ulang candidate menjadi `best.pt` sebelum review.

In [ ]:
from google.colab import files
for artifact in (CANDIDATE_OUTPUT, AUDIT_PATH, COMPOSITION_PATH, INTERNAL_METRICS, BASELINE_GOLD, CANDIDATE_GOLD, COMPARISON_PATH):
    if artifact.is_file():
        files.download(str(artifact))
print('Selesai. backend/weights/best.pt tidak pernah disentuh oleh notebook ini.')

## Interpretasi hasil

- Candidate hanya layak dilanjutkan bila gate pilot lolos dan preview prediksi pada footage target tidak menunjukkan regresi nyata.
- Jika helmet/vest turun, kurangi atau nonaktifkan specialist single-class; jangan sekadar menambah epoch.
- Jika gloves/shoes tetap rendah, prioritas berikutnya adalah data full-body/CCTV beranotasi lengkap dari domain kamera target, bukan lebih banyak close-up produk.
- Perbedaan kecil dari baseline historis dapat terjadi karena versi Ultralytics/GPU; gunakan hasil baseline yang direproduksi dalam run yang sama untuk delta.
- `gloves` dan `shoes` tetap indikator eksperimental sampai gate produksi yang disepakati benar-benar lolos.